# Task 2.1 — Dataset Selection and Setup

## Random Seeds


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import sys
sys.path.insert(0, '.')

# ---- RANDOM SEED (set at top for reproducibility) ----
SEED = 42
np.random.seed(SEED)
rng = np.random.RandomState(SEED)

print("Random seed set:", SEED)
print("numpy version:", np.__version__)

Random seed set: 42
numpy version: 1.26.4


## Dataset Justification

**What the dataset is:**  
A synthetically generated binary classification dataset where each training example is a 2D multivariate Gaussian distribution. Two classes of 100 distributions each are created, with 20 samples drawn per distribution.

**Why it is a reasonable testbed:**  
The SMM's core contribution is learning from *distributions* rather than individual points. This dataset is specifically designed to test this: Class +1 distributions are horizontally elongated (covariance `[[1.8, 0.05],[0.05, 0.25]]`) and Class -1 are vertically elongated (covariance `[[0.25, 0.05],[0.05, 1.8]]`). Crucially, the distribution **means fully overlap** (both drawn from N(0, 0.6·I)), so any method using only means (SVM) will perform at chance. The SMM, which embeds the full distribution via the expected kernel (Eq. 4), captures the covariance difference and correctly classifies. This directly mirrors the claim in Section 6.1 of the paper.

**Limitations compared to the paper's dataset:**  
- Paper uses 10D distributions (ours: 2D) and 500 per class (ours: 100)  
- Paper's class separation comes from differences in *both* mean and covariance; ours is a pure covariance-difference problem (more extreme but cleaner)  
- Our smaller dataset may show higher variance in cross-validation accuracy  
- 2D simplification means the advantage of SMM over ASVM is less pronounced than in 10D

**Preprocessing:**  
No preprocessing is required. Data is generated directly from known distributions. Samples are already zero-centered and unit-scaled per the generation parameters.


In [ ]:
# Dataset generation
N_PER_CLASS = 100   # distributions per class
N_SAMPLES   = 20    # samples drawn per distribution
D = 2               # 2D for visualization

distributions, labels, means_list = [], [], []

# Class +1: Horizontally elongated Gaussians
for _ in range(N_PER_CLASS):
    mean = rng.randn(D) * 0.6
    cov  = np.array([[1.8, 0.05], [0.05, 0.25]])
    samples = rng.multivariate_normal(mean, cov, size=N_SAMPLES)
    distributions.append(samples)
    labels.append(+1)
    means_list.append(samples.mean(axis=0))

# Class -1: Vertically elongated Gaussians
for _ in range(N_PER_CLASS):
    mean = rng.randn(D) * 0.6
    cov  = np.array([[0.25, 0.05], [0.05, 1.8]])
    samples = rng.multivariate_normal(mean, cov, size=N_SAMPLES)
    distributions.append(samples)
    labels.append(-1)
    means_list.append(samples.mean(axis=0))

labels    = np.array(labels)
means_arr = np.array(means_list)

print(f"Total distributions: {len(distributions)}")
print(f"Samples per distribution: {N_SAMPLES}")
print(f"Features (D): {D}")
print(f"Class balance: {np.sum(labels==1)} positive, {np.sum(labels==-1)} negative")
print(f"\nMean of class +1 means: {means_arr[:N_PER_CLASS].mean(axis=0).round(3)}")
print(f"Mean of class -1 means: {means_arr[N_PER_CLASS:].mean(axis=0).round(3)}")
print("(Means overlap — SVM trained on means will fail)")

Total distributions: 200
Samples per distribution: 20
Features (D): 2
Class balance: 100 positive, 100 negative

Mean of class +1 means: [-0.042  0.023]
Mean of class -1 means: [0.012 -0.011]
(Means overlap — SVM trained on means will fail)


## Dataset Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Covariance-Structured Dataset (Task 2.1)', fontsize=12, fontweight='bold')

ax = axes[0]
for i in [0, 5, 10, 20, 30]:
    ax.scatter(distributions[i][:,0], distributions[i][:,1], c='steelblue', alpha=0.3, s=12)
    c = np.cov(distributions[i].T)
    ev, evec = np.linalg.eigh(c)
    ang = np.degrees(np.arctan2(evec[1,0], evec[0,0]))
    ax.add_patch(Ellipse(distributions[i].mean(0), 3*np.sqrt(ev[1]), 3*np.sqrt(ev[0]),
                         angle=ang, edgecolor='steelblue', facecolor='none', lw=1.5))
for i in [N_PER_CLASS, N_PER_CLASS+5, N_PER_CLASS+10, N_PER_CLASS+20]:
    ax.scatter(distributions[i][:,0], distributions[i][:,1], c='tomato', alpha=0.3, s=12)
    c = np.cov(distributions[i].T)
    ev, evec = np.linalg.eigh(c)
    ang = np.degrees(np.arctan2(evec[1,0], evec[0,0]))
    ax.add_patch(Ellipse(distributions[i].mean(0), 3*np.sqrt(ev[1]), 3*np.sqrt(ev[0]),
                         angle=ang, edgecolor='tomato', facecolor='none', lw=1.5))
ax.scatter([], [], c='steelblue', label='Class +1 (horizontal)', s=40)
ax.scatter([], [], c='tomato', label='Class -1 (vertical)', s=40)
ax.legend(); ax.set_title('5 Sample Distributions per Class'); ax.set_xlim(-5,5); ax.set_ylim(-7,7)

ax2 = axes[1]
ax2.scatter(means_arr[:N_PER_CLASS,0], means_arr[:N_PER_CLASS,1], c='steelblue', alpha=0.5, s=20, label='Class +1')
ax2.scatter(means_arr[N_PER_CLASS:,0], means_arr[N_PER_CLASS:,1], c='tomato', alpha=0.5, s=20, label='Class -1')
ax2.set_title('Distribution Means (fully overlapping)'); ax2.legend()
plt.tight_layout()
plt.savefig('partB/results/task2_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to partB/results/task2_dataset.png")

Plot saved to partB/results/task2_dataset.png
